# Introduction to Data Science — ภาคปฏิบัติ บท 07–08
ข้อมูลทุกแถวเป็นข้อมูลจำลอง ไม่มีข้อมูลบุคคลจริง วาง CSV ทั้งสองไฟล์ไว้โฟลเดอร์เดียวกับ notebook

คำตอบของคุณ: เพิ่มโค้ด/ผลรัน/คำอธิบายใต้ TODO โดยคงลำดับ train → validation → final test
P01/P03 ใช้ final test เมื่อเลือกแบบเสร็จแล้วเท่านั้น; P04 เป็นส่วนเสริม

แพ็กเกจที่ใช้: numpy, pandas, scikit-learn, matplotlib (ติดตั้งในสภาพแวดล้อม Python ของผู้เรียนก่อนรัน เช่น `python -m pip install numpy pandas scikit-learn matplotlib`)


In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print({'python': sys.version.split()[0], 'numpy': np.__version__,
       'pandas': pd.__version__, 'sklearn': sklearn.__version__})


## P01 — Simple และ Multiple Linear Regression
CSV: `temp_forecast_c` = อุณหภูมิพยากรณ์ก่อนวันใช้งาน (°C), `working_day` = วันทำการ 1 / วันหยุด 0, `energy_kwh` = พลังงานจริง (kWh)
Simple ใช้ temp_forecast_c; Multiple เพิ่ม working_day
รายงาน validation MAE/RMSE (kWh) และ R² ของทั้งสองแบบ อธิบาย score และเลือกด้วย validation RMSE


In [ ]:
development = pd.read_csv('regression_development.csv', parse_dates=['date'])
train = development.iloc[:72].copy()
validation = development.iloc[72:].copy()
assert train.date.max() < validation.date.min()
assert len(train) == 72 and len(validation) == 18
display(development.head())
# TODO: สร้าง, fit, predict, ประเมิน Simple และ Multiple บน validation
# เก็บ final test ไว้จนเลือกแบบเสร็จใน P03


## P02 — Bias–Variance experiment
Protocol กลางสำหรับเทียบผล: x ~ Uniform(0,6), y = sin(x)+Normal(0,0.25), train 35 จุด
validation 200 จุดใช้ seed 2026 คงเดิมทุกโมเดล/รอบ; train seeds 11,22,33; degree 1,3,12
ใช้ PolynomialFeatures(include_bias=False) → StandardScaler → LinearRegression และเก็บ prediction ที่ x=2.0
บันทึก RMSE train/validation; สรุปจากผลจริง รวมข้อจำกัดเมื่อใช้เพียง 3 รอบ


In [ ]:
def sine_data(seed, n):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 6, size=(n, 1))
    y = np.sin(x[:, 0]) + rng.normal(0, 0.25, n)
    return x, y

x_val, y_val = sine_data(2026, 200)
seeds, degrees = [11, 22, 33], [1, 3, 12]
# TODO: วนทุก seed × degree; fit 35 จุด แล้วเก็บผลครบ 9 แถว
# ค่า RMSE = np.sqrt(mean_squared_error(y_true, y_pred))


## P03 — Residual และปรับปรุงโมเดล
ตรวจ residual ของ Multiple model บน validation 18 แถวเทียบ fitted values และวันที่
เปรียบเทียบ A/B/C/D ตามโจทย์: A = simple linear features 2 ตัว; B เพิ่ม temp²; C/D ใช้ features B แต่ StandardScaler → Ridge(alpha=0.1/10)
เลือกด้วย validation RMSE; หากต่าง <0.01 kWh เลือกแบบง่ายกว่า แล้ว alpha ต่ำกว่าเมื่อโครงสร้างเท่ากัน
หลังเลือกแล้วจึง refit บน development 90 แถวและประเมิน final test ครั้งสุดท้าย ไม่เลือกแบบใหม่จาก test


In [ ]:
# TODO: plots, feature engineering, pipelines, validation table, model choice
# หลังเลือกแบบแล้วเท่านั้น:
# final_test = pd.read_csv('regression_final_test.csv', parse_dates=['date'])
# TODO: refit แบบที่เลือกบน development; รายงาน final-test MAE, RMSE, R²


## P04 — ส่วนเสริม bootstrap
ใช้ model A: temp_forecast_c + working_day จาก development90 สุ่มคู่ X,y ไม่แยกกัน
ordinary pairs: สุ่ม 90 แถวใส่คืน; moving-block pairs: เริ่มบล็อกจาก 0..83, ยาว7วัน สุ่มบล็อกใส่คืนให้ ≥90 แล้วตัดเหลือ90
ทำ 300 รอบ/วิธี, แต่ละวิธีเริ่ม RNG seed20260922, fit LinearRegression ทุกครั้งและเก็บ coefficient ของ temp
รายงาน percentile 2.5/97.5; อธิบาย dependence, ความหมาย coefficient และข้อจำกัดการเรียกช่วง confidence vs prediction


In [ ]:
# TODO: bootstrap 2 วิธี, ช่วง percentile, คำอธิบายสมมติฐาน
# Hint for moving blocks:
# starts = rng.integers(0, n - block_length + 1, size=int(np.ceil(n / block_length)))
# indices = np.concatenate([np.arange(s, s + block_length) for s in starts])[:n]


## สรุปผลด้วยภาษาของตนเอง
- สิ่งที่ model ทำได้บนข้อมูลนี้และหลักฐานตัวเลข
- สิ่งที่ยังสรุปไม่ได้ (เช่น เหตุและผล หรือผลในระบบจริง)
- รายการไฟล์/ผลรันที่ส่ง และเครื่องมือช่วยที่ใช้ (ถ้ามี)
